# 🧪 Test Technique — Data Engineering Pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tanouar/1to1code/blob/main/tests-techniques/test_data_engineering.ipynb)

**Durée :** 1h30  
**Date :** _______________  &nbsp;&nbsp; **Prénom / Nom :** _______________

---

## 📋 Consignes générales

- Répondez uniquement dans les cellules marquées ✏️.
- Chaque cellule doit être exécutable sans erreur.
- Les imports nécessaires sont à votre charge.
- **Aucune aide extérieure autorisée.**

Les fichiers de données sont hébergés sur GitHub.  
Vous devez les **télécharger ou les lire directement** via leur URL avec `requests` / `pandas`.

| Fichier | URL |
|---|---|
| Commandes (JSON) | `https://raw.githubusercontent.com/tanouar/1to1code/main/data/orders.json` |
| Lignes EDI | `https://raw.githubusercontent.com/tanouar/1to1code/main/data/shipping.edi` |
| Produits (Parquet) | `https://raw.githubusercontent.com/tanouar/1to1code/main/data/products.parquet` |
| Base Northwind | `https://raw.githubusercontent.com/tanouar/1to1code/main/data/northwind.db` |


In [ ]:
# ⚙️ Installation des dépendances
!pip install duckdb pyarrow requests -q
print("✅ Dépendances installées")


---

# 📊 Exercice 1 — Pipeline de Données Python

Vous allez travailler avec **trois sources de données hétérogènes** :

| Source | Format | Clé de jointure |
|---|---|---|
| Commandes clients | JSON (URL GitHub) | `order_id` |
| Lignes de commande | EDI 856 simplifié (URL GitHub) | `order_id`, `product_id` |
| Catalogue produits | Parquet (URL GitHub) | `product_id` |

L'objectif est de **fusionner ces trois tables** pour obtenir une vue complète des ventes, puis d'effectuer une analyse statistique de base.

> **Format EDI utilisé :** Les segments de données sont des lignes commençant par `LIN*`  
> Chaque segment suit ce schéma : `LIN*<order_id>*<product_id>*<quantity>*<unit_price>*`  
> ⚠️ Certaines valeurs peuvent être **manquantes** (champ vide entre deux `*`).


## 1.1 — Chargement des commandes (JSON)

Téléchargez le fichier JSON depuis l'URL suivante et chargez-le dans un **DataFrame pandas** nommé `df_orders`.

```
https://raw.githubusercontent.com/tanouar/1to1code/main/data/orders.json
```

*Conseil : utilisez `requests` puis `pd.DataFrame(response.json())`,  
ou directement `pd.read_json(url)`.*

Affichez les premières lignes et le nombre total de lignes.


In [ ]:
# ✏️ Votre code ici


## 1.2 — Chargement des lignes de commande (EDI)

**Étape A :** Téléchargez le fichier EDI depuis l'URL suivante et récupérez le texte brut.

```
https://raw.githubusercontent.com/tanouar/1to1code/main/data/shipping.edi
```

Affichez les 10 premières lignes pour comprendre la structure.

**Étape B :** Parsez les segments `LIN*` pour construire un **DataFrame pandas** nommé `df_edi`  
avec les colonnes : `order_id`, `product_id`, `quantity`, `unit_price_edi`.  
*Conseil : utilisez `.split('\n')` puis `.startswith('LIN*')` pour filtrer les lignes.*


In [ ]:
# ✏️ Étape A — Récupération du texte EDI brut


In [ ]:
# ✏️ Étape B — Parsing des segments LIN* → df_edi


## 1.3 — Chargement du catalogue produits (Parquet)

Téléchargez le fichier Parquet depuis l'URL suivante et chargez-le dans un **DataFrame pandas** nommé `df_products`.

```
https://raw.githubusercontent.com/tanouar/1to1code/main/data/products.parquet
```

*Conseil : utilisez `requests` + `io.BytesIO` puis `pd.read_parquet()`.*

Affichez son contenu.


In [ ]:
# ✏️ Votre code ici


## 1.4 — Nettoyage des données

Avant de fusionner, inspectez et nettoyez chaque DataFrame.

**Pour `df_orders` :**
- Vérifiez les **doublons** sur `order_id` et supprimez-les
- Convertissez la colonne `order_date` en type **datetime**

**Pour `df_edi` :**
- Identifiez les **valeurs manquantes** dans `unit_price_edi`
- Remplacez les prix manquants par le prix du **catalogue produits** (`df_products`)

*Créez `df_orders_clean` et `df_edi_clean` pour conserver les versions nettoyées.*


In [ ]:
# ✏️ Nettoyage de df_orders → df_orders_clean


In [ ]:
# ✏️ Nettoyage de df_edi → df_edi_clean


## 1.5 — Fusion des trois tables

Réalisez les jointures dans l'ordre suivant :
1. `df_orders_clean` ⊕ `df_edi_clean` → clé commune : `order_id`
2. Résultat ⊕ `df_products` → clé commune : `product_id` (colonnes : `product_name`, `category`, `in_stock`)

Ajoutez une colonne calculée `total_line = quantity × unit_price_final` (arrondie à 2 décimales).

Stockez le résultat dans `df_final`. Affichez sa forme et les 10 premières lignes.


In [ ]:
# ✏️ Votre code ici


## 1.6 — Analyse statistique

À partir de `df_final`, calculez et affichez :

1. **Chiffre d'affaires total par catégorie de produits** (triée décroissante)
2. **Nombre de commandes uniques par pays** (triée décroissante)
3. **Top 5 des produits les plus vendus** par quantité totale
4. **Valeur moyenne, max et min** d'une commande (agrégée par `order_id`)


In [ ]:
# ✏️ Votre code ici


---

# 🦆 Exercice 2 — SQL avec DuckDB + Northwind

Vous allez interroger la base de données **Northwind** — un jeu de données classique simulant les ventes d'une société de distribution alimentaire.

**Tables principales à utiliser :**

| Table | Description |
|---|---|
| `Orders` | Entêtes de commandes (date, client, ...) |
| `OrderDetails` | Lignes de commande (produit, quantité, prix, remise) |
| `Customers` | Clients (pays, ville, ...) |
| `Products` | Produits (nom, prix unitaire, ...) |
| `Categories` | Catégories de produits |



## 2.1 — Téléchargement de la base Northwind

Utilisez une **commande Linux** (`wget`) pour télécharger le fichier `northwind.db` depuis l'URL suivante :

```
https://raw.githubusercontent.com/tanouar/1to1code/main/data/northwind.db
```

Sauvegardez le fichier sous le nom `northwind.db` et vérifiez sa taille avec `ls -lh`.


In [ ]:
# ✏️ Commande wget ici


## 2.2 — Connexion DuckDB et exploration

1. Importez `duckdb` et créez une connexion **en mémoire**
2. Installez et chargez l'extension **sqlite** de DuckDB
3. Attachez `northwind.db` comme base de données nommée `nw` avec le type `sqlite`
4. Listez toutes les tables disponibles


In [ ]:
# ✏️ Votre code ici


## 2.3 — Requête 1 : Top 10 clients par chiffre d'affaires

Écrivez une requête SQL qui affiche, **pour chaque client**, le chiffre d'affaires total et le segment de client. Les seuils pour la segmentation sont les suivants :
- **Segment "VIP"** : CA ≥ 10 000  
- **Segment "Régulier"** : 3 000 ≤ CA < 10 000
- **Segment "Occasionnel"** : CA < 3 000

**Colonnes attendues :** `CustomerName`, `CA`, `Segment`

**Calcul du CA :** `Quantity × UnitPrice`  
**Tables à joindre :** `Orders`, `Customers`, `OrderDetails`, `Products`
**Tri :** par `CA` décroissant — **Limiter aux 10 premiers clients**


In [ ]:
# ✏️ Votre code ici


## 2.4 — Requête 2 : Catégorie avec le CA le plus élevé

**Tables :** `OrderDetails` · `Products` · `Categories`

**Consigne :** Afficher le **nom de la catégorie** et son **chiffre d'affaires total** pour la catégorie ayant généré le CA le plus élevé.

**Colonnes attendues :**

| Colonne | Description |
|---|---|
| `CategoryName` | Nom de la catégorie |
| `ChiffreAffaires` | CA total arrondi à 2 décimales |


In [ ]:
# ✏️ Votre code ici


## 2.5 — BONUS : Moyenne glissante sur les ventes mensuelles

Écrivez une requête SQL qui calcule, **pour chaque mois**, le chiffre d'affaires mensuel et la **moyenne glissante sur 3 mois** (mois courant + 2 mois précédents).

**Colonnes attendues :** `mois` (format `YYYY-MM`), `revenu_mensuel`, `moyenne_glissante_3_mois`

*Conseil : utilisez une CTE (`WITH`) pour calculer les revenus mensuels, puis une **window function** `AVG(...) OVER (ORDER BY mois ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)`.*


In [ ]:
# ✏️ Votre code ici


---

*Fin du test. Vérifiez que toutes vos cellules s'exécutent sans erreur avant de rendre le notebook.*
